# KBP Demo — Knowledge Boundary Probe

This notebook walks through the two core workflows from the paper:
1. **H1**: Training a linear probe to detect knowledge sufficiency
2. **H2**: Computing effective rank as a label-free proxy

All steps run without a GPU using synthetic data so you can verify the pipeline locally.
Swap `make_synthetic_hidden_states()` for real extracted states from `scripts/extract_hidden_states.py`.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

print('KBP Demo — ready')

## 1. Synthetic Hidden States

We simulate hidden states at three different layers:
- Layer 10 (early): weak separability
- Layer 23 (ℓ*): strong separability — paper optimum
- Layer 30 (late): slightly weaker than ℓ*

In [ ]:
def make_synthetic_hidden_states(n=1000, d=512, n_layers=32, best_layer=23, seed=42):
    """Simulate hidden states with layer-dependent separability."""
    rng = np.random.RandomState(seed)
    
    # KS/KD labels (1 = sufficient, 0 = deficient)
    labels = np.array([1] * (n // 2) + [0] * (n // 2))
    
    hidden_states = {}
    for layer in range(0, n_layers, 2):  # sample every 2nd layer
        # Separability peaks at best_layer (Gaussian envelope)
        signal_strength = np.exp(-0.5 * ((layer - best_layer) / 6) ** 2)
        
        H = rng.randn(n, d) * 0.3  # background noise
        # Signal: KS queries cluster around +direction, KD around -direction
        direction = rng.randn(d)
        direction /= np.linalg.norm(direction)
        H += np.outer(labels * 2 - 1, direction) * signal_strength * 2
        
        # L2 normalize (as paper)
        H = H / (np.linalg.norm(H, axis=1, keepdims=True) + 1e-8)
        hidden_states[layer] = H
    
    return hidden_states, labels

hidden_states, labels = make_synthetic_hidden_states()
print(f'Generated hidden states at {len(hidden_states)} layers')
print(f'Shape per layer: {hidden_states[0].shape}')
print(f'Labels: {labels.sum()} KS, {(1-labels).sum()} KD')

## 2. H1: Layer-Wise Probe Training

In [ ]:
from kbp.probe import LayerWiseProbeTrainer

trainer = LayerWiseProbeTrainer(n_seeds=3, train_ratio=0.70)
lw_results = trainer.fit_all_layers(
    hidden_states, labels, total_layers=32, model_name='Synthetic-LM'
)

best_layer, best_auroc = lw_results.best_layer()
print(f'\nBest layer: {best_layer} (AUROC={best_auroc:.4f})')
print(f'Expected:  layer 22 or 24 (near ℓ*=23)')

In [ ]:
# Figure 1 reproduction
layers = sorted(lw_results.results.keys())
depths = [l / 32 for l in layers]
means = [lw_results.mean_auroc(l) for l in layers]
stds  = [lw_results.std_auroc(l) for l in layers]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(depths, means, 'o-', color='#2196F3', lw=2, ms=5, label='KBP Linear Probe')
ax.fill_between(depths,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                alpha=0.15, color='#2196F3')
ax.axhline(0.803, color='gray', ls='--', lw=1.5, label='Truthfulness Probe baseline (paper)')
ax.axvline(best_layer/32, color='red', ls=':', lw=1.5, label=f'Best layer (ℓ*={best_layer})')
ax.axvspan(0.60, 0.75, alpha=0.08, color='gold', label='Paper-predicted peak zone')
ax.set_xlabel('Layer depth (normalized)', fontsize=12)
ax.set_ylabel('AUROC', fontsize=12)
ax.set_title('Figure 1: Layer-wise Probe AUROC', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, loc='lower right')
ax.set_xlim(0, 1); ax.set_ylim(0.4, 1.0)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figure1_demo.png', dpi=150)
plt.show()
print('Saved to results/figure1_demo.png')

## 3. H2: Effective Rank Analysis

In [ ]:
from kbp.effective_rank import EffectiveRankEstimator, compute_effective_rank

# Simulate 6 tasks with varying knowledge levels
def make_task(n=256, d=512, rank=5, seed=0):
    """Low rank = KS, high rank = KD."""
    rng = np.random.RandomState(seed)
    if rank < d:
        W = rng.randn(rank, d)
        H = rng.randn(n, rank) @ W
    else:
        H = rng.randn(n, d)
    H /= np.linalg.norm(H, axis=1, keepdims=True) + 1e-8
    return H

tasks = {
    'PopQA-High (KS)':   make_task(rank=4),
    'MMLU-General (KS)': make_task(rank=6, seed=1),
    'PopQA-Mid (Mixed)': make_task(rank=20, seed=2),
    'MKQA-Arabic (KD)':  make_task(rank=80, seed=3),
    'MedBench (KD)':     make_task(rank=150, seed=4),
    'LaoBench (KD)':     make_task(rank=400, seed=5),
}

estimator = EffectiveRankEstimator(n_queries=256, n_bootstrap=10)

print(f'{'Task':<26} {'erank':>8} {'Regime':>10} {'SFT':>20}')
print('-' * 68)
task_eranks = {}
for name, H in tasks.items():
    result = estimator.estimate(H)
    task_eranks[name] = result.erank
    sft = '✓ viable' if result.sft_recommended else '✗ need data'
    print(f'{name:<26} {result.erank:>8.2f} {result.regime:>10} {sft:>20}')

In [ ]:
# Figure 2: erank vs. (mock) gradient SNR
# Paper: ρs = -0.82 (strong negative correlation)
eranks = list(task_eranks.values())
# Mock SNR: inversely proportional to erank (log scale)
snrs = [np.exp(8 - 0.15 * er + np.random.randn() * 0.3) for er in eranks]

from scipy import stats
rho, pval = stats.spearmanr(eranks, snrs)

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#2196F3', '#2196F3', '#FF9800', '#F44336', '#F44336', '#F44336']
for i, (name, er, snr) in enumerate(zip(tasks.keys(), eranks, snrs)):
    ax.scatter(er, snr, c=colors[i], s=80, zorder=3)
    ax.annotate(name.split(' ')[0], (er, snr), xytext=(5, 3),
                textcoords='offset points', fontsize=9)

# OLS fit
z = np.polyfit(eranks, np.log(snrs), 1)
x_fit = np.linspace(min(eranks)-1, max(eranks)+1, 100)
ax.plot(x_fit, np.exp(np.polyval(z, x_fit)), 'k--', lw=1.5, alpha=0.6)

ax.set_yscale('log')
ax.set_xlabel('Activation Effective Rank', fontsize=12)
ax.set_ylabel('Gradient SNR (log scale)', fontsize=12)
ax.set_title(f'Figure 2: Effective Rank vs. Gradient SNR\n'
             f'Spearman ρs={rho:.2f} (p={pval:.3f})', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figure2_demo.png', dpi=150)
plt.show()
print(f'Spearman ρs = {rho:.2f} (paper target: ≈ -0.82)')

## 4. KBP Inference (End-to-End)

With a real model, replace `mock_predict` with `kbp.predict()`.

In [ ]:
from kbp.probe import LinearProbe
from kbp.kbp import KBPResult

# Train a probe on the best layer
X_train, X_test, y_train, y_test = train_test_split(
    hidden_states[best_layer], labels, train_size=0.7, stratify=labels, random_state=0
)
probe = LinearProbe()
probe.fit(X_train, y_train)

def mock_predict(probe, hidden_state_row):
    """Simulate KBP.predict() for a single query."""
    score = probe.predict_proba(hidden_state_row.reshape(1, -1))[0]
    margin = probe.margin(hidden_state_row.reshape(1, -1))[0]
    label = 'KNOWLEDGE_SUFFICIENT' if score >= 0.5 else 'KNOWLEDGE_DEFICIENT'
    return KBPResult(label=label, score=float(score), margin=float(margin))

# Demo on 5 test examples
print('KBP Inference Demo')
print('=' * 60)
for i in range(5):
    result = mock_predict(probe, X_test[i])
    truth = 'KS' if y_test[i] == 1 else 'KD'
    correct = '✓' if result.is_knowledge_sufficient == (y_test[i] == 1) else '✗'
    print(f'[{correct}] {result.label:<28} score={result.score:.3f}  '
          f'margin={result.margin:.3f}  truth={truth}')

print(f'\nTest AUROC: {probe.auroc(X_test, y_test):.4f}')

## 5. Margin → Prompt Sensitivity

Queries near the decision boundary (low margin) are more sensitive to paraphrasing.

In [ ]:
# Simulate K paraphrases per query by adding small noise
K = 8
rng = np.random.RandomState(42)

margins = probe.margin(X_test)
variances = []
for i in range(len(X_test)):
    # K paraphrases: small perturbations in the hidden state space
    paraphrases = X_test[i:i+1] + rng.randn(K, X_test.shape[1]) * 0.05
    # Re-normalize
    paraphrases /= np.linalg.norm(paraphrases, axis=1, keepdims=True)
    scores = probe.predict_proba(paraphrases)
    variances.append(scores.var())

variances = np.array(variances)

from scipy.stats import spearmanr
rho_mv, pval_mv = spearmanr(margins, variances)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(margins, variances, alpha=0.3, s=8, c='#2196F3')
ax.set_xlabel('Decision boundary margin', fontsize=12)
ax.set_ylabel('Detection variance across paraphrases', fontsize=12)
ax.set_title(f'Figure 3: Margin vs. Prompt Sensitivity\n'
             f'Spearman ρs = {rho_mv:.2f} (p={pval_mv:.2e})', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figure3_demo.png', dpi=150)
plt.show()
print(f'Paper target: ρs ≈ -0.63 to -0.71')

## Summary

| Component | Location | Key result |
|-----------|----------|------------|
| H1: Linear probe | `kbp/probe.py` | AUROC 88.4 at ℓ*=23 |
| H2: Effective rank | `kbp/effective_rank.py` | ρs ≈ -0.82 with gradient SNR |
| RAG routing | `kbp/routing.py` | 51% retrieval, matches full-RAG accuracy |
| SFT viability | `kbp/effective_rank.py` | 11/12 tasks correct |
| Cross-model | `kbp/kbp.py:align_to()` | AUROC 77-79 after Procrustes |

To run with a real model:

```bash
python scripts/extract_hidden_states.py \
    --model meta-llama/Meta-Llama-3-8B \
    --dataset popqa \
    --output data/hidden_states/llama3_popqa.pt

python scripts/train_probe.py \
    --hidden-states data/hidden_states/llama3_popqa.pt \
    --layer 23 \
    --output checkpoints/kbp_llama3.pkl

python experiments/run_h1.py \
    --hidden-states data/hidden_states/llama3_popqa.pt \
    --n-layers 32 --seeds 5 --plot
```